In [65]:
import pandas as pd
import PyPDF2
import re
import os

# Ruta al archivo que descargaste
ruta_archivo = r"C:\Users\Carla\Curso_reskilling_Data\Sprint-10\BMX_L.XPT"  # cambia el nombre si el tuyo es distinto

# Leer el archivo .XPT (formato SAS transport)
df = pd.read_sas(ruta_archivo, format='xport')

# Mostrar las primeras filas
print(df.head())

# Mostrar las columnas disponibles
print("\nColumnas disponibles:", df.columns.tolist())


       SEQN  BMDSTATS  BMXWT  BMIWT  BMXRECUM  BMIRECUM  BMXHEAD  BMIHEAD  \
0  130378.0       1.0   86.9    NaN       NaN       NaN      NaN      NaN   
1  130379.0       1.0  101.8    NaN       NaN       NaN      NaN      NaN   
2  130380.0       1.0   69.4    NaN       NaN       NaN      NaN      NaN   
3  130381.0       1.0   34.3    NaN       NaN       NaN      NaN      NaN   
4  130382.0       3.0   13.6    NaN       NaN       1.0      NaN      NaN   

   BMXHT  BMIHT  ...  BMXLEG  BMILEG  BMXARML  BMIARML  BMXARMC  BMIARMC  \
0  179.5    NaN  ...    42.8     NaN     42.0      NaN     35.7      NaN   
1  174.2    NaN  ...    38.5     NaN     38.7      NaN     33.7      NaN   
2  152.9    NaN  ...    38.5     NaN     35.5      NaN     36.3      NaN   
3  120.1    NaN  ...     NaN     NaN     25.4      NaN     23.4      NaN   
4    NaN    1.0  ...     NaN     NaN      NaN      1.0      NaN      1.0   

   BMXWAIST  BMIWAIST  BMXHIP  BMIHIP  
0      98.3       NaN   102.9     NaN  


In [66]:
# Cargar archivo demográfico
df_demo = pd.read_sas(r"C:\Users\Carla\Curso_reskilling_Data\Sprint-10/DEMO_L.XPT", format='xport')

# Unir con el de medidas usando el ID de participante
df_merged = pd.merge(df, df_demo[['SEQN', 'RIAGENDR']], on='SEQN')

# Renombrar columna para que sea legible
df_merged['Sexo'] = df_merged['RIAGENDR'].map({1: 'Hombre', 2: 'Mujer'})

# Verifica
print(df_merged[['SEQN', 'Sexo']].head())


       SEQN    Sexo
0  130378.0  Hombre
1  130379.0  Hombre
2  130380.0   Mujer
3  130381.0   Mujer
4  130382.0  Hombre


Voy a filtrar por las medidas que me intersan y eliminar los nan para poder sacar promedios minimos y maximos 
BMXWT = Peso en Kilogramos
BMXHT = Altura en centimetros
BMXWAIST = circunferencia de cintura en cm
BMXHIP = circunferencia de cadera en cm

In [67]:
df_filtrado = df[['BMXWT', 'BMXHT', 'BMXWAIST', 'BMXHIP']].dropna()


In [68]:
print(df_filtrado.describe())

             BMXWT        BMXHT     BMXWAIST       BMXHIP
count  6749.000000  6749.000000  6749.000000  6749.000000
mean     80.551322   166.736079    98.124952   106.206979
std      22.405777    10.037981    17.938101    14.595127
min      27.900000   131.500000    54.600000    69.900000
25%      64.400000   159.500000    85.000000    96.400000
50%      77.500000   166.300000    97.000000   103.700000
75%      93.100000   174.000000   109.600000   113.400000
max     248.200000   200.700000   187.000000   187.100000


In [69]:
df_medidas_sexo = df_merged[['BMXWT', 'BMXHT', 'BMXWAIST', 'BMXHIP','Sexo']].dropna()

In [70]:
df_cuerpos_USA = df_medidas_sexo.rename(columns={
    "BMXWT": "Peso",
    "BMXHT": "Altura",
    "BMXWAIST": "Cintura",
    "BMXHIP": "Cadera"
})
df_cuerpos_USA

,Peso,Altura,Cintura,Cadera,Sexo
0,86.9,179.5,98.3,102.9,Hombre
1,101.8,174.2,114.7,112.4,Hombre
2,69.4,152.9,93.5,98.0,Mujer
5,90.6,173.3,106.1,110.6,Hombre
6,103.5,155.9,122.0,148.9,Mujer
...,...,...,...,...,...
8853,48.9,162.4,69.6,81.4,Hombre
8854,60.4,151.4,89.0,96.4,Mujer
8857,79.3,173.3,98.4,97.7,Hombre
8858,81.9,179.1,96.0,103.3,Hombre


# Extraccion de tallas europeas

### Mango

In [71]:
# Leer el archivo PDF
ruta_pdf = r'C:\Users\Carla\Curso_reskilling_Data\Sprint-10\Jeans rectos tiro alto - Mujer _ MANGO España (Península y Baleares).pdf'
with open(ruta_pdf, 'rb') as archivo:
    lector = PyPDF2.PdfReader(archivo)
    texto = ''
    for pagina in lector.pages:
        texto += pagina.extract_text()

# Usar regex para encontrar líneas con patrón: número número número
patron = r'(\d{2})\s+(\d{2,3})\s+(\d{2,3})'
coincidencias = re.findall(patron, texto)

# Crear un DataFrame si se encuentran coincidencias
if coincidencias:
    df_mango = pd.DataFrame(coincidencias, columns=["Talla_EU", "Cintura", "Cadera"])
    df_mango[["Talla_EU", "Cintura", "Cadera"]] = df_mango[["Talla_EU", "Cintura", "Cadera"]].astype(int)

    # Añadir columna ID
    df_mango["ID"] = ["Mango_" + str(t) for t in df_mango["Talla_EU"]]

    # Opcional: añadir marca y nombre del producto
    df_mango["Marca"] = "Mango"
    df_mango["Producto"] = "Jeans rectos tiro alto"



else:
    print(" No se encontraron tallas con el patrón esperado.")

columnas = ["ID"] + [col for col in df_mango.columns if col != "ID"]
df_mango = df_mango[columnas]

columnas = ["ID", "Talla_EU", "Cintura", "Cadera", "Marca", "Producto"]

df_mango

,ID,Talla_EU,Cintura,Cadera,Marca,Producto
0,Mango_32,32,59,86,Mango,Jeans rectos tiro alto
1,Mango_34,34,62,90,Mango,Jeans rectos tiro alto
2,Mango_36,36,66,94,Mango,Jeans rectos tiro alto
3,Mango_38,38,70,98,Mango,Jeans rectos tiro alto
4,Mango_40,40,74,102,Mango,Jeans rectos tiro alto
5,Mango_42,42,78,106,Mango,Jeans rectos tiro alto
6,Mango_44,44,85,112,Mango,Jeans rectos tiro alto
7,Mango_46,46,92,118,Mango,Jeans rectos tiro alto
8,Mango_48,48,97,122,Mango,Jeans rectos tiro alto
9,Mango_50,50,102,126,Mango,Jeans rectos tiro alto


### Zara

Para leer el Pdf de Zara y que sea equivalente con el de Mango hay que multiplicar por 2 las tallas de cintura y cadera 

porque Zara tiene las medidas en formato plano (te muestran la mitad de la medida)

In [72]:
# Ruta al nuevo PDF de Zara
ruta_pdf = r'C:\Users\Carla\Curso_reskilling_Data\Sprint-10\JEANS Z1975 STRAIGHT TIRO ALTO LONG LENGTH - Azul claro _ ZARA_02 España.pdf'

# Leer PDF
with open(ruta_pdf, 'rb') as archivo:
    lector = PyPDF2.PdfReader(archivo)
    texto = ''
    for pagina in lector.pages:
        texto += pagina.extract_text()

# Separar en líneas
lineas = texto.strip().split('\n')

# Extraer todas las líneas con 6 valores numéricos
bloque_medidas = []
for linea in lineas:
    numeros = linea.strip().split()
    if len(numeros) == 6 and all(n.replace('.', '', 1).isdigit() for n in numeros):
        bloque_medidas.append([float(n) for n in numeros])

# Verificar que hay al menos 4 filas (tallas + 3 medidas)
if len(bloque_medidas) >= 4:
    tallas = [int(n) for n in bloque_medidas[0]]
    cintura = [x * 2 for x in bloque_medidas[1]]
    cadera = [x * 2 for x in bloque_medidas[2]]


    df_zara = pd.DataFrame({
        "Talla_EU": tallas,
        "Cintura": cintura,
        "Cadera": cadera,
        "Marca": "Zara",
        "Producto": "Jeans Z1975 STRAIGHT TIRO ALTO LONG LENGTH"
    })

    df_zara["ID"] = df_zara["Marca"] + "_" + df_zara["Talla_EU"].astype(str)
    columnas_ordenadas = ["ID", "Talla_EU", "Cintura", "Cadera", "Marca", "Producto"]
    df_zara = df_zara[columnas_ordenadas]


else:
    print(" No se pudieron organizar los datos. Verifica el contenido.")


df_zara


,ID,Talla_EU,Cintura,Cadera,Marca,Producto
0,Zara_32,32,59.0,87.0,Zara,Jeans Z1975 STRAIGHT TIRO ALTO LONG LENGTH
1,Zara_34,34,63.0,91.0,Zara,Jeans Z1975 STRAIGHT TIRO ALTO LONG LENGTH
2,Zara_36,36,67.0,95.0,Zara,Jeans Z1975 STRAIGHT TIRO ALTO LONG LENGTH
3,Zara_38,38,71.0,99.0,Zara,Jeans Z1975 STRAIGHT TIRO ALTO LONG LENGTH
4,Zara_40,40,75.0,103.0,Zara,Jeans Z1975 STRAIGHT TIRO ALTO LONG LENGTH
5,Zara_42,42,79.0,107.0,Zara,Jeans Z1975 STRAIGHT TIRO ALTO LONG LENGTH


### H&M

In [73]:
# Ruta a la carpeta donde están los archivos PDF de H&M
carpeta = r'C:\Users\Carla\Curso_reskilling_Data\Sprint-10'
archivos_hm = sorted([os.path.join(carpeta, f) for f in os.listdir(carpeta) if f.endswith(".pdf") and "H&M" in f])

# Lista para acumular DataFrames individuales
dfs = []

for archivo in archivos_hm:
    with open(archivo, 'rb') as f:
        lector = PyPDF2.PdfReader(f)
        texto = ''
        for pagina in lector.pages:
            texto += pagina.extract_text()

    lineas = texto.strip().split("\n")

    try:
        # Extraer tallas desde línea 6
        tallas = re.findall(r'\d+', lineas[6])
        tallas = [int(t) for t in tallas if 30 <= int(t) <= 70]
        n = len(tallas)

        # Extraer medidas
        cintura = re.findall(r'\d{2,3}', lineas[7])
        cadera = re.findall(r'\d{2,3}', lineas[8])
        entrepierna = re.findall(r'\d{2,3}(?:\.\d)?', lineas[14])

        if len(cintura) >= n and len(cadera) >= n and len(entrepierna) >= n:
            df = pd.DataFrame({
                "ID": [f"H&M_{t}" for t in tallas],
                "Talla_EU": tallas,
                "Cintura": [int(x) for x in cintura[:n]],
                "Cadera": [int(x) for x in cadera[:n]],
                "Marca": "H&M",
                "Producto": "Straight High Ankle Jeans"
            })
            dfs.append(df)
        else:
            print(f"⚠️ Medidas incompletas en: {os.path.basename(archivo)}")

    except Exception as e:
        print(f"❌ Error procesando {os.path.basename(archivo)}: {e}")

# Combinar todos los DataFrames en uno solo
df_hm_total = pd.concat(dfs, ignore_index=True)



df_hm_total

,ID,Talla_EU,Cintura,Cadera,Marca,Producto
0,H&M_30,30,59,78,H&M,Straight High Ankle Jeans
1,H&M_32,32,62,82,H&M,Straight High Ankle Jeans
2,H&M_34,34,62,82,H&M,Straight High Ankle Jeans
3,H&M_36,36,66,90,H&M,Straight High Ankle Jeans
4,H&M_38,38,70,94,H&M,Straight High Ankle Jeans
5,H&M_40,40,70,94,H&M,Straight High Ankle Jeans
6,H&M_42,42,78,100,H&M,Straight High Ankle Jeans
7,H&M_44,44,82,103,H&M,Straight High Ankle Jeans
8,H&M_46,46,82,103,H&M,Straight High Ankle Jeans
9,H&M_48,48,93,110,H&M,Straight High Ankle Jeans


# Extraccion de tallas estadounidenses

### American Eagle

In [74]:
# Ruta local al archivo PDF de American Eagle
ruta_pdf = r'C:\Users\Carla\Curso_reskilling_Data\Sprint-10\AmericanEagle Super High-Waisted Wide-Leg Jean holgado stretch con drapeado de ensueño.pdf'

# Extraer texto del PDF
with open(ruta_pdf, 'rb') as archivo:
    lector = PyPDF2.PdfReader(archivo)
    texto = ''
    for pagina in lector.pages:
        texto += pagina.extract_text()

# Usar expresión regular para extraer talla numérica, cintura y cadera en cm
patron = r'(\d{1,3})\s+(\d{2,3})\s+(\d{2,3})'
coincidencias = re.findall(patron, texto)

# Crear DataFrame y limpiar
if coincidencias:
    df_ae = pd.DataFrame(coincidencias, columns=["Talla_USA", "Cintura", "Cadera"])
    df_ae = df_ae.astype({"Talla_USA": str, "Cintura": int, "Cadera": int})
    
    # Agregar columnas extra
    df_ae["Marca"] = "American Eagle"
    df_ae["Producto"] = "Super High-Waisted Wide-Leg Jean"
    df_ae["ID"] = df_ae["Marca"] + "_" + df_ae["Talla_USA"]

    # Reordenar columnas
    columnas_ordenadas = ["ID", "Talla_USA", "Cintura", "Cadera", "Marca", "Producto"]
    df_ae = df_ae[columnas_ordenadas]

   
else:
    print(" No se encontraron datos con el formato esperado.")

In [75]:
# Cuando hago la extraccion se generan datos dobles por lo que filtro por el primero que es el que necesito
df_ae_unico = df_ae.drop_duplicates(subset="Talla_USA", keep="first")

# Reordenar columnas por estética
df_ae_unico = df_ae_unico[["ID", "Talla_USA", "Cintura", "Cadera", "Marca", "Producto"]]

# Mapeo manual de talla US a EU
mapa_us_eu = {
    "000": 28,
    "00": 30,
    "0": 32,
    "2": 34,
    "4": 36,
    "6": 38,
    "8": 40,
    "10": 42,
    "12": 44,
    "14": 46,
    "16": 48,
    "18": 50,
    "20": 52
}

# Crear nueva columna con la conversión
df_ae_unico["Talla_EU"] = df_ae_unico["Talla_USA"].astype(str).map(mapa_us_eu)

# Reordenar columnas para que 'Talla_EU' esté al lado de 'Talla'
columnas_ordenadas = ["ID", "Talla_USA", "Talla_EU", "Cintura", "Cadera", "Marca", "Producto"]
df_ae_unico = df_ae_unico[columnas_ordenadas]


df_ae_unico

,ID,Talla_USA,Talla_EU,Cintura,Cadera,Marca,Producto
0,American Eagle_000,000,28,57,81,American Eagle,Super High-Waisted Wide-Leg Jean
1,American Eagle_00,00,30,60,84,American Eagle,Super High-Waisted Wide-Leg Jean
2,American Eagle_2,2,34,30,155,American Eagle,Super High-Waisted Wide-Leg Jean
3,American Eagle_0,0,32,62,86,American Eagle,Super High-Waisted Wide-Leg Jean
4,American Eagle_4,4,36,32,155,American Eagle,Super High-Waisted Wide-Leg Jean
6,American Eagle_6,6,38,34,160,American Eagle,Super High-Waisted Wide-Leg Jean
8,American Eagle_8,8,40,36,160,American Eagle,Super High-Waisted Wide-Leg Jean
13,American Eagle_10,10,42,75,99,American Eagle,Super High-Waisted Wide-Leg Jean
15,American Eagle_12,12,44,79,103,American Eagle,Super High-Waisted Wide-Leg Jean
17,American Eagle_14,14,46,83,107,American Eagle,Super High-Waisted Wide-Leg Jean


### Aeropostale

In [76]:

# Ruta al archivo PDF local
ruta_pdf = r'C:\Users\Carla\Curso_reskilling_Data\Sprint-10\High-Waisted Wide Leg Jean Aeropostale.pdf'

# Leer el texto del PDF
texto = ''
with open(ruta_pdf, 'rb') as archivo:
    lector = PyPDF2.PdfReader(archivo)
    for pagina in lector.pages:
        texto += pagina.extract_text()

# Extraer las tallas y medidas (manual por estructura no tabular)
tallas = ["000", "00-0", "2-4", "6-8", "10-12", "14-16", "18-20"]
waist_in = [23, 24.5, 26.5, 28.5, 31.25, 34.75, 38.25]
hips_in = [33, 34.5, 36.5, 38.5, 41.25, 44.75, 48.25]

# Crear DataFrame
df_aeropostale = pd.DataFrame({
    "Talla_US": tallas,
    "Waist_in": waist_in,
    "Hips_in": hips_in
})

# Convertir a centímetros
df_aeropostale["Waist_cm"] = df_aeropostale["Waist_in"] * 2.54
df_aeropostale["Hips_cm"] = df_aeropostale["Hips_in"] * 2.54

# Añadir talla europea aproximada
mapa_talla_eu = {
    "000": "30",
    "00-0": "32",
    "2-4": "34",
    "6-8": "36",
    "10-12": "38",
    "14-16": "40",
    "18-20": "42"
}
df_aeropostale["Talla_EU"] = df_aeropostale["Talla_US"].map(mapa_talla_eu)

# Agregar marca, producto e ID
df_aeropostale["Marca"] = "Aeropostale"
df_aeropostale["Producto"] = "High-Waisted Wide Leg Jean"
df_aeropostale["ID"] = df_aeropostale["Marca"] + "_" + df_aeropostale["Talla_EU"]

# Reordenar columnas
columnas_finales = ["ID", "Talla_US", "Talla_EU", "Waist_cm", "Hips_cm", "Marca", "Producto"]
df_aeropostale = df_aeropostale[columnas_finales]

# Renombrar columnas
df_aeropostale = df_aeropostale.rename(columns={
    "Talla_US": "Talla_USA",
    "Waist_cm": "Cintura",
    "Hips_cm": "Cadera"
})

# Reordenar columnas
columnas_finales = ["ID", "Talla_USA", "Talla_EU", "Cintura", "Cadera", "Marca", "Producto"]
df_aeropostale = df_aeropostale[columnas_finales]


# Mostrar resultado
df_aeropostale


,ID,Talla_USA,Talla_EU,Cintura,Cadera,Marca,Producto
0,Aeropostale_30,000,30,58.420,83.820,Aeropostale,High-Waisted Wide Leg Jean
1,Aeropostale_32,00-0,32,62.230,87.630,Aeropostale,High-Waisted Wide Leg Jean
2,Aeropostale_34,2-4,34,67.310,92.710,Aeropostale,High-Waisted Wide Leg Jean
3,Aeropostale_36,6-8,36,72.390,97.790,Aeropostale,High-Waisted Wide Leg Jean
4,Aeropostale_38,10-12,38,79.375,104.775,Aeropostale,High-Waisted Wide Leg Jean
5,Aeropostale_40,14-16,40,88.265,113.665,Aeropostale,High-Waisted Wide Leg Jean
6,Aeropostale_42,18-20,42,97.155,122.555,Aeropostale,High-Waisted Wide Leg Jean


### Old Navy 

In [77]:
# Ruta al archivo PDF
ruta_pdf = r'C:\Users\Carla\Curso_reskilling_Data\Sprint-10\High-Waisted Wow Wide-Leg Jeans _ Old Navy.pdf'

# Mapeo de tallas US a EU
conversion_talla = {
    "00": 30, "0": 32, "2": 34, "4": 36, "6": 38,
    "8": 40, "10": 42, "12": 44, "14": 46, "16": 48,
    "18": 50, "20": 52, "22": 54, "24": 56
}

# Función para convertir rango a promedio
def promedio_cm(valores):
    nums = list(map(int, valores))
    return round(sum(nums) / len(nums))

# Leer texto del PDF
with open(ruta_pdf, 'rb') as archivo:
    lector = PyPDF2.PdfReader(archivo)
    texto = ''
    for pagina in lector.pages:
        texto += pagina.extract_text()

# Extraer datos con regex
patron = r'(\d{1,2}(?:\s\d{1,2})?)\s+(\d{2,3}(?:\s\d{2,3})?)\s+(\d{2,3}(?:\s\d{2,3})?)'
coincidencias = re.findall(patron, texto)

# Crear DataFrame
datos = []
for talla_us, cintura_raw, cadera_raw in coincidencias:
    cintura_cm = promedio_cm(cintura_raw.split())
    cadera_cm = promedio_cm(cadera_raw.split())
    # Si la talla es un rango (como "8 10"), tomamos el primer número para conversión
    talla_simple = talla_us.split()[0]
    talla_eu = conversion_talla.get(talla_simple, None)
    
    datos.append({
        "ID": f"OldNavy_{talla_us.strip().replace(' ', '-')}",
        "Talla_USA": talla_us.strip(),
        "Talla_EU": talla_eu,
        "Cintura": cintura_cm,
        "Cadera": cadera_cm,
        "Marca": "Old Navy",
        "Producto": "High-Waisted Wow Wide-Leg Jeans"
    })

df_oldnavy = pd.DataFrame(datos)

# Mostrar resultado
df_oldnavy

,ID,Talla_USA,Talla_EU,Cintura,Cadera,Marca,Producto
0,OldNavy_00,00,30.0,61,88,Old Navy,High-Waisted Wow Wide-Leg Jeans
1,OldNavy_0-2,0 2,32.0,65,92,Old Navy,High-Waisted Wow Wide-Leg Jeans
2,OldNavy_4-6,4 6,36.0,70,96,Old Navy,High-Waisted Wow Wide-Leg Jeans
3,OldNavy_8-10,8 10,40.0,75,102,Old Navy,High-Waisted Wow Wide-Leg Jeans
4,OldNavy_12-14,12 14,44.0,82,110,Old Navy,High-Waisted Wow Wide-Leg Jeans
5,OldNavy_16-18,16 18,48.0,96,121,Old Navy,High-Waisted Wow Wide-Leg Jeans
6,OldNavy_20,20,52.0,107,131,Old Navy,High-Waisted Wow Wide-Leg Jeans
7,OldNavy_20-22,20 22,52.0,110,134,Old Navy,High-Waisted Wow Wide-Leg Jeans
8,OldNavy_24-26,24 26,56.0,121,146,Old Navy,High-Waisted Wow Wide-Leg Jeans
9,OldNavy_28-30,28 30,NaN,134,158,Old Navy,High-Waisted Wow Wide-Leg Jeans


Después de ordenar los datos ahora agrego todo en un dataframe europa y un dataframe Estados Unidos 

In [78]:
df_ae_unico["Región"] = "EEUU"
df_oldnavy["Región"] = "EEUU"
df_aeropostale["Región"] = "EEUU"

# Concatenar todo
df_usa_final = pd.concat([df_ae_unico, df_oldnavy, df_aeropostale], ignore_index=True)
df_usa_final

,ID,Talla_USA,Talla_EU,Cintura,Cadera,Marca,Producto,Región
0,American Eagle_000,000,28,57.000,81.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
1,American Eagle_00,00,30,60.000,84.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
2,American Eagle_2,2,34,30.000,155.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
3,American Eagle_0,0,32,62.000,86.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
4,American Eagle_4,4,36,32.000,155.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
5,American Eagle_6,6,38,34.000,160.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
6,American Eagle_8,8,40,36.000,160.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
7,American Eagle_10,10,42,75.000,99.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
8,American Eagle_12,12,44,79.000,103.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU
9,American Eagle_14,14,46,83.000,107.000,American Eagle,Super High-Waisted Wide-Leg Jean,EEUU


In [79]:
df_mango["Región"] = "EUROPA"
df_zara["Región"] = "EUROPA"
df_hm_total["Región"] = "EUROPA"

# Concatenar todo
df_europa = pd.concat([df_mango, df_zara, df_hm_total], ignore_index=True)
df_europa

,ID,Talla_EU,Cintura,Cadera,Marca,Producto,Región
0,Mango_32,32,59.0,86.0,Mango,Jeans rectos tiro alto,EUROPA
1,Mango_34,34,62.0,90.0,Mango,Jeans rectos tiro alto,EUROPA
2,Mango_36,36,66.0,94.0,Mango,Jeans rectos tiro alto,EUROPA
3,Mango_38,38,70.0,98.0,Mango,Jeans rectos tiro alto,EUROPA
4,Mango_40,40,74.0,102.0,Mango,Jeans rectos tiro alto,EUROPA
5,Mango_42,42,78.0,106.0,Mango,Jeans rectos tiro alto,EUROPA
6,Mango_44,44,85.0,112.0,Mango,Jeans rectos tiro alto,EUROPA
7,Mango_46,46,92.0,118.0,Mango,Jeans rectos tiro alto,EUROPA
8,Mango_48,48,97.0,122.0,Mango,Jeans rectos tiro alto,EUROPA
9,Mango_50,50,102.0,126.0,Mango,Jeans rectos tiro alto,EUROPA
